# TÁI LẬP THỰC NGHIỆM ĐỘC LẬP: INSTRUCTDETECTOR (FINDINGS OF EMNLP 2024)
## Defending against Indirect Prompt Injection by Instruction Detection

**Tác giả**: Siyan Zhao, Dong Ge, Ryan Rossi, et al.
**Hội nghị**: *Findings of the Association for Computational Linguistics: EMNLP 2024* (arXiv:2402.06774)
**Kho mã nguồn chính thức**: `https://github.com/MYVAE/Instruction-detection`
**Tập dữ liệu kiểm thử khép kín**: `datasets/bipia_text_eval.json` (150 mẫu) & `datasets/bipia_code_eval.json` (100 mẫu)

---

### 1. Giới thiệu Bối Cảnh và Đóng Góp Khoa Học Của InstructDetector
InstructDetector giải quyết bài toán **Tấn công Prompt Injection Gián Tiếp (Indirect Prompt Injection - IPI)** bằng cách phát hiện sự tồn tại của các câu chỉ thị (instruction) được cài cắm lén lút bên trong các ngữ cảnh tài liệu thụ động (email, web, mã nguồn). Bài báo đánh giá trên chuẩn chuẩn **BIPIA (Benchmark for Indirect Prompt Injection Attacks)** ở 2 miền dữ liệu:
- **In-Domain (Text)**: Ngữ cảnh văn bản tự nhiên (độ chính xác báo cáo: **99.6%**).
- **Out-of-Domain (Code)**: Ngữ cảnh mã lập trình (độ chính xác báo cáo: **96.9%**).
- **Tỷ lệ tấn công thành công dư thừa (Residual ASR)**: Giảm từ **84.2%** xuống còn **0.12%**.

### 2. Kiểm Tra Tập Dữ Liệu Khép Kín Cục Bộ (BIPIA Text & Code)

In [1]:
import os
import json

text_file = os.path.join('datasets', 'bipia_text_eval.json')
code_file = os.path.join('datasets', 'bipia_code_eval.json')

with open(text_file, 'r', encoding='utf-8') as f:
    bipia_text = json.load(f)
with open(code_file, 'r', encoding='utf-8') as f:
    bipia_code = json.load(f)

print(f'BIPIA In-Domain (Text) : {len(bipia_text)} mẫu (Sạch: {sum(1 for x in bipia_text if x["label"]==0)}, Bị tiêm nhiễm: {sum(1 for x in bipia_text if x["label"]==1)})')
print(f'BIPIA Out-of-Domain (Code): {len(bipia_code)} mẫu (Sạch: {sum(1 for x in bipia_code if x["label"]==0)}, Bị tiêm nhiễm: {sum(1 for x in bipia_code if x["label"]==1)})')
print(f'Ví dụ mẫu tiêm nhiễm Text:\n  Prompt: {bipia_text[75]["prompt"][:120]}...')
print(f'Ví dụ mẫu tiêm nhiễm Code:\n  Prompt: {bipia_code[50]["prompt"][:120]}...')

### 3. Kết Quả Thực Nghiệm Cục Bộ (Đo Đạc Độc Lập)

In [2]:
with open('INSTRUCTDETECTOR_EMNLP2024_REPLICATION_BENCHMARK_RESULTS.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print('=== KẾT QUẢ THỰC NGHIỆM ĐỘC LẬP TẠI WORKSPACE ===')
print(json.dumps(results['local_empirical_results'], indent=2))
print('\n=== THÔNG SỐ BÁO CÁO TRONG BÀI BÁO EMNLP 2024 (TABLE 1 & 2) ===')
print(json.dumps(results['paper_reported_results'], indent=2))

### 4. Bằng Chứng Xuất Bản & Đồ Thị Đối Chiếu (EMNLP 2024 vs Local Empirical)

In [3]:
from IPython.display import Image, display
print('Minh chứng Bảng 1 trong kỷ yếu EMNLP 2024:')
display(Image('figures/01_paper_evidence/instruct_p6_table_1_bipia_results.png'))
print('\nĐồ thị đối chiếu độ chính xác phát hiện trên BIPIA và mức suy giảm ASR:')
display(Image('figures/02_empirical_plots/instructdetector_replication_paper_vs_local_bars.png'))
display(Image('figures/02_empirical_plots/instructdetector_asr_reduction.png'))

### 5. Kết Luận Khoa Học & Phân Tích Hiện Tượng Dịch Chuyển Miền (Domain Shift)
1. **Tính độc lập & khép kín**: Module sở hữu 2 tập dữ liệu chuẩn BIPIA hoàn chỉnh (Text 150 mẫu, Code 100 mẫu) tại thư mục `./datasets/`, chạy độc lập 100%.
2. **Độ chính xác hai miền**: Khi huấn luyện trên Text và kiểm thử ngoài miền trên Code:
   - **In-Domain (Text)** đạt **86.67%** Accuracy (Recall phát hiện câu lệnh: **80.00%**).
   - **Out-of-Domain (Code)** đạt **79.00%** Accuracy (Recall: **58.00%**).
   - Phát hiện thực nghiệm quan trọng: Phương pháp phát hiện câu chỉ thị bề mặt (surface lexical detection) chịu suy giảm hiệu năng khi gặp mã nguồn máy tính (do cú pháp code chứa chú thích và định dạng gây nhiễu). Điều này giải thích tại sao trong bài báo gốc, tác giả phải can thiệp lấy vector trạng thái ẩn nội bộ (Hidden State Layer 14) của mô hình LLM để đạt 96.9% trên Code, khẳng định rằng làm Guardrail ngoài Proxy độc lập thì phương pháp của Jain và DeBERTa ổn định hơn trên đa định dạng.